<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/ColumnValueCount_problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from functools import reduce

spark = SparkSession.builder.getOrCreate()

# Input Data
data = [
    (10, 20, 11, 20),
    (20, 11, 10, 99),
    (10, 11, 20, 1),
    (30, 12, 20, 99),
    (10, 11, 20, 20),
    (40, 13, 15, 3),
    (30, 8, 11, 99)
]

df = spark.createDataFrame(data, ["A", "B", "C", "D"])

df.show()

+---+---+---+---+
|  A|  B|  C|  D|
+---+---+---+---+
| 10| 20| 11| 20|
| 20| 11| 10| 99|
| 10| 11| 20|  1|
| 30| 12| 20| 99|
| 10| 11| 20| 20|
| 40| 13| 15|  3|
| 30|  8| 11| 99|
+---+---+---+---+



specific **Case**





Soluton#**1**

In [11]:
from pyspark.sql.functions.builtin import date_from_unix_date
windowa=Window.orderBy(col("A").desc())
windowb=Window.orderBy(col("B").desc())
windowc=Window.orderBy(col("C").desc())
windowd=Window.orderBy(col("D").desc())
df_a=df.select("A").groupBy("A").count().withColumnRenamed("count", "count_a").withColumn("rank_a", row_number().over(windowa))
df_b=df.select("B").groupBy("B").count().withColumnRenamed("count", "count_b").withColumn("rank_b", row_number().over(windowb))
df_c=df.select("C").groupBy("C").count().withColumnRenamed("count", "count_c").withColumn("rank_c", row_number().over(windowc))
df_d=df.select("D").groupBy("D").count().withColumnRenamed("count", "count_d").withColumn("rank_d", row_number().over(windowd))
final_df=df_a.join(df_b, df_a.rank_a==df_b.rank_b, "full").join(df_c, df_a.rank_a==df_c.rank_c, "full").join(df_d, df_a.rank_a==df_d.rank_d, "full").select("A","count_a", "B", "count_b", "C","count_c", "D" , "count_d").orderBy("A", "B", "C", "D")
final_df.show()

+----+-------+---+-------+----+-------+----+-------+
|   A|count_a|  B|count_b|   C|count_c|   D|count_d|
+----+-------+---+-------+----+-------+----+-------+
|NULL|   NULL|  8|      1|NULL|   NULL|NULL|   NULL|
|  10|      3| 11|      3|  10|      1|   1|      1|
|  20|      1| 12|      1|  11|      2|   3|      1|
|  30|      2| 13|      1|  15|      1|  20|      2|
|  40|      1| 20|      1|  20|      3|  99|      3|
+----+-------+---+-------+----+-------+----+-------+



Solution#2

In [36]:

df_final = None

for c in df.columns:
  window_spec = Window.orderBy(col(c).desc())
  if c=='A':
    df_final = df.select(c).groupBy(c).count().withColumnRenamed("count", f"count_{c}").withColumn(f"rank_{c}", row_number().over(window_spec))
  else:
    df_c = df.select(c).groupBy(c).count().withColumnRenamed("count", f"count_{c}").withColumn(f"rank_{c}", row_number().over(window_spec))
    df_final=df_final.join(df_c, df_c[f'rank_{c}'] == df_final.rank_A, "full")


In [40]:
 column_without_rank=[col_name for col_name in df_final.columns if not col_name.startswith('rank_')]
 df_final.select(*column_without_rank).show()

+----+-------+---+-------+----+-------+----+-------+
|   A|count_A|  B|count_B|   C|count_C|   D|count_D|
+----+-------+---+-------+----+-------+----+-------+
|NULL|   NULL|  8|      1|NULL|   NULL|NULL|   NULL|
|  40|      1| 20|      1|  20|      3|  99|      3|
|  30|      2| 13|      1|  15|      1|  20|      2|
|  20|      1| 12|      1|  11|      2|   3|      1|
|  10|      3| 11|      3|  10|      1|   1|      1|
+----+-------+---+-------+----+-------+----+-------+



**Genralization**  Solution #3

In [21]:
c = df.columns
dfs_with_ranks = []

print("Number of columns are ", c)

# Generate a list of DataFrames, each with a column, its count, and its rank
for col_name in c:
    window_spec = Window.orderBy(col(col_name).desc())
    df_col = (
        df.select(col_name)
        .groupBy(col_name)
        .count()
        .withColumnRenamed("count", f"count_{col_name}")
        .withColumn(f"rank_{col_name}", row_number().over(window_spec))
    )
    dfs_with_ranks.append(df_col)

# Initialize the final DataFrame with the first grouped DataFrame
final_joined_df = dfs_with_ranks[0]
first_col_name = c[0]
first_rank_col_name = f"rank_{first_col_name}"

# Perform subsequent joins using the rank of the first column as the common join key
for i in range(1, len(dfs_with_ranks)):
    current_df = dfs_with_ranks[i]
    current_col_name = c[i]
    current_rank_col_name = f"rank_{current_col_name}"

    final_joined_df = final_joined_df.join(
        current_df,
        final_joined_df[first_rank_col_name] == current_df[current_rank_col_name],
        "full"
    )

# Dynamically select the columns: original column and its count, then order
select_cols = []
order_cols = []
for col_name in c:
    select_cols.append(col_name)
    select_cols.append(f"count_{col_name}")
    order_cols.append(col_name)

final_joined_df = final_joined_df.select(*select_cols).orderBy(*order_cols)
final_joined_df.show()

Number of columns are  ['A', 'B', 'C', 'D']
+----+-------+---+-------+----+-------+----+-------+
|   A|count_A|  B|count_B|   C|count_C|   D|count_D|
+----+-------+---+-------+----+-------+----+-------+
|NULL|   NULL|  8|      1|NULL|   NULL|NULL|   NULL|
|  10|      3| 11|      3|  10|      1|   1|      1|
|  20|      1| 12|      1|  11|      2|   3|      1|
|  30|      2| 13|      1|  15|      1|  20|      2|
|  40|      1| 20|      1|  20|      3|  99|      3|
+----+-------+---+-------+----+-------+----+-------+

